# Machine Learning: Churn Prediction Pipeline
**Input:** data/gold/telco_gold.parquet  
**Models:** Logistic Regression → Random Forest → XGBoost  
**Output:** models/xgb_churn_model.pkl · reports/at_risk_customers.csv  
**Goal:** Predict which customers will churn and explain why

In [1]:
# Run this cell first to install any missing libraries
import subprocess
import sys

libraries = ['xgboost', 'shap', 'imbalanced-learn']
for lib in libraries:
    subprocess.check_call([sys.executable, '-m', 'pip', 
                          'install', lib, '-q'])

print("✓ All libraries ready")

✓ All libraries ready


In [2]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')  
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score,
    classification_report,
    confusion_matrix,
    RocCurveDisplay
)
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
import shap
import joblib
import os

print("✓ All libraries imported successfully")

✓ All libraries imported successfully


In [3]:
# Load the Gold layer — our ML-ready feature engineered data
df = pd.read_parquet('C:/Users/DELL/Documents/telecom-churn-analytics/data/gold/telco_gold.parquet')

print(f"✓ Gold data loaded")
print(f"  Shape   : {df.shape}")
print(f"  Columns : {df.shape[1]}")
print(f"\nFirst 3 rows:")
df.head(3)

✓ Gold data loaded
  Shape   : (7043, 28)
  Columns : 28

First 3 rows:


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,OnlineSecurity,OnlineBackup,...,tenure_group,avg_monthly_spend,service_count,Contract_One year,Contract_Two year,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,InternetService_Fiber optic,InternetService_No
0,7590-VHVEG,Female,0,1,0,1,0,0,0,1,...,0-1 Year,14.92,0,False,False,False,True,False,False,False
1,5575-GNVDE,Male,0,0,0,34,1,0,1,0,...,2-4 Years,53.99,0,True,False,False,False,True,False,False
2,3668-QPYBK,Male,0,0,0,2,1,0,1,1,...,0-1 Year,36.05,0,False,False,False,False,True,False,False


In [4]:
print("All columns in Gold layer:")
for i, col in enumerate(df.columns, 1):
    print(f"  {i:2}. {col}")

All columns in Gold layer:
   1. customerID
   2. gender
   3. SeniorCitizen
   4. Partner
   5. Dependents
   6. tenure
   7. PhoneService
   8. MultipleLines
   9. OnlineSecurity
  10. OnlineBackup
  11. DeviceProtection
  12. TechSupport
  13. StreamingTV
  14. StreamingMovies
  15. PaperlessBilling
  16. MonthlyCharges
  17. TotalCharges
  18. Churn
  19. tenure_group
  20. avg_monthly_spend
  21. service_count
  22. Contract_One year
  23. Contract_Two year
  24. PaymentMethod_Credit card (automatic)
  25. PaymentMethod_Electronic check
  26. PaymentMethod_Mailed check
  27. InternetService_Fiber optic
  28. InternetService_No


In [5]:
# Columns to drop — not useful for prediction
# customerID: just an identifier, no predictive value
# Churn: this IS what we're predicting — can't use it as input
# tenure_group: we already have tenure as a number, 
#               the text category version causes issues

drop_cols = ['customerID', 'Churn', 'tenure_group']

# Drop only columns that actually exist in the dataframe
# (safety check in case some columns have different names)
existing_drop = [c for c in drop_cols if c in df.columns]

X = df.drop(columns=existing_drop)
y = df['Churn']

print(f"✓ Features (X) shape : {X.shape}")
print(f"✓ Target   (y) shape : {y.shape}")
print(f"\nTarget distribution:")
print(f"  Not churned (0): {(y==0).sum():,} ({(y==0).mean()*100:.1f}%)")
print(f"  Churned     (1): {(y==1).sum():,} ({(y==1).mean()*100:.1f}%)")
print(f"\nFeature columns ({X.shape[1]} total):")
print(list(X.columns))

✓ Features (X) shape : (7043, 25)
✓ Target   (y) shape : (7043,)

Target distribution:
  Not churned (0): 5,174 (73.5%)
  Churned     (1): 1,869 (26.5%)

Feature columns (25 total):
['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'PaperlessBilling', 'MonthlyCharges', 'TotalCharges', 'avg_monthly_spend', 'service_count', 'Contract_One year', 'Contract_Two year', 'PaymentMethod_Credit card (automatic)', 'PaymentMethod_Electronic check', 'PaymentMethod_Mailed check', 'InternetService_Fiber optic', 'InternetService_No']


In [6]:
# Check for any remaining non-numeric columns
# (One-hot encoding should have handled most, 
#  but gender and some others may still be text)
non_numeric = X.select_dtypes(include=['object', 'category']).columns.tolist()
print(f"Non-numeric columns found: {non_numeric}")

if non_numeric:
    # Convert remaining text columns using one-hot encoding
    X = pd.get_dummies(X, columns=non_numeric, drop_first=True)
    print(f"✓ Encoded {len(non_numeric)} columns")
    print(f"✓ New X shape: {X.shape}")
else:
    print("✓ All columns are already numeric — ready for ML")

Non-numeric columns found: ['gender']
✓ Encoded 1 columns
✓ New X shape: (7043, 25)


In [7]:
# Split data: 80% for training, 20% for testing
# stratify=y ensures both splits have same 73.5/26.5 ratio
# random_state=42 makes results reproducible 
# (same split every time you run this)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"✓ Train/Test split complete")
print(f"  Training set   : {X_train.shape[0]:,} rows ({X_train.shape[0]/len(X)*100:.0f}%)")
print(f"  Test set       : {X_test.shape[0]:,} rows ({X_test.shape[0]/len(X)*100:.0f}%)")
print(f"\n  Train - Not churned: {(y_train==0).sum():,} | Churned: {(y_train==1).sum():,}")
print(f"  Test  - Not churned: {(y_test==0).sum():,}  | Churned: {(y_test==1).sum():,}")

✓ Train/Test split complete
  Training set   : 5,634 rows (80%)
  Test set       : 1,409 rows (20%)

  Train - Not churned: 4,139 | Churned: 1,495
  Test  - Not churned: 1,035  | Churned: 374


In [8]:
# Apply SMOTE only on TRAINING data — never on test data
# Why not test data? Because test data must reflect real-world 
# distribution. SMOTE is only for helping the model learn.

smote = SMOTE(random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

print(f"✓ SMOTE applied to training data")
print(f"\n  BEFORE SMOTE (training set):")
print(f"    Not churned: {(y_train==0).sum():,}")
print(f"    Churned    : {(y_train==1).sum():,}")
print(f"\n  AFTER SMOTE (training set):")
print(f"    Not churned: {(y_train_bal==0).sum():,}")
print(f"    Churned    : {(y_train_bal==1).sum():,}")
print(f"\n  Training set size: {len(y_train):,} → {len(y_train_bal):,}")

✓ SMOTE applied to training data

  BEFORE SMOTE (training set):
    Not churned: 4,139
    Churned    : 1,495

  AFTER SMOTE (training set):
    Not churned: 4,139
    Churned    : 4,139

  Training set size: 5,634 → 8,278


In [9]:
# StandardScaler transforms every column to have 
# mean=0 and std=1. This is needed for Logistic Regression
# because it's sensitive to the scale of numbers.
# (tenure goes 0-72, MonthlyCharges goes 18-119 — 
#  without scaling, larger numbers dominate unfairly)

# XGBoost and Random Forest don't need scaling,
# but we prepare it now for Logistic Regression

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_bal)
X_test_scaled  = scaler.transform(X_test)

print("✓ Feature scaling complete")
print(f"  Before scaling - tenure mean    : {X_train_bal['tenure'].mean():.1f}")
print(f"  Before scaling - MonthlyCharges : {X_train_bal['MonthlyCharges'].mean():.1f}")

# Convert scaled arrays back to DataFrames for easier handling
X_train_scaled = pd.DataFrame(
    X_train_scaled, 
    columns=X_train_bal.columns
)
X_test_scaled = pd.DataFrame(
    X_test_scaled, 
    columns=X_test.columns
)
print(f"  After scaling  - tenure mean    : {X_train_scaled['tenure'].mean():.4f}")
print(f"  After scaling  - MonthlyCharges : {X_train_scaled['MonthlyCharges'].mean():.4f}")

✓ Feature scaling complete
  Before scaling - tenure mean    : 28.1
  Before scaling - MonthlyCharges : 68.2
  After scaling  - tenure mean    : 0.0000
  After scaling  - MonthlyCharges : 0.0000


## Model 1: Logistic Regression — Baseline
The simplest possible model. Always start here.
If a complex model doesn't beat this, something is wrong.

In [10]:
print("Training Logistic Regression...")

# Train the model
lr_model = LogisticRegression(
    max_iter=1000,    # max iterations to find the best line
    random_state=42
)
lr_model.fit(X_train_scaled, y_train_bal)

# Predict probabilities on test set
# predict_proba returns probability of being class 0 and class 1
# We take [:, 1] which is the probability of churning (class 1)
lr_probs = lr_model.predict_proba(X_test_scaled)[:, 1]
lr_preds = lr_model.predict(X_test_scaled)

# Calculate ROC-AUC score
lr_auc = roc_auc_score(y_test, lr_probs)

print(f"\n✓ Logistic Regression trained")
print(f"  ROC-AUC Score : {lr_auc:.4f}")
print(f"\nClassification Report:")
print(classification_report(y_test, lr_preds, 
      target_names=['Not Churned', 'Churned']))

Training Logistic Regression...

✓ Logistic Regression trained
  ROC-AUC Score : 0.8205

Classification Report:
              precision    recall  f1-score   support

 Not Churned       0.86      0.81      0.84      1035
     Churned       0.56      0.64      0.60       374

    accuracy                           0.77      1409
   macro avg       0.71      0.73      0.72      1409
weighted avg       0.78      0.77      0.77      1409



## Model 2: Random Forest
Builds 100 decision trees and takes a majority vote.
Uses the balanced training data but doesn't need scaling.

In [11]:
print("Training Random Forest (100 trees)...")
print("This may take 30-60 seconds...")

rf_model = RandomForestClassifier(
    n_estimators=100,   # number of trees
    random_state=42,
    n_jobs=-1           # use all CPU cores to speed up
)
rf_model.fit(X_train_bal, y_train_bal)

# Predict on unscaled test data 
# (Random Forest doesn't need scaling)
rf_probs = rf_model.predict_proba(X_test)[:, 1]
rf_preds = rf_model.predict(X_test)

rf_auc = roc_auc_score(y_test, rf_probs)

print(f"\n✓ Random Forest trained")
print(f"  ROC-AUC Score : {rf_auc:.4f}")
print(f"\nClassification Report:")
print(classification_report(y_test, rf_preds,
      target_names=['Not Churned', 'Churned']))

Training Random Forest (100 trees)...
This may take 30-60 seconds...

✓ Random Forest trained
  ROC-AUC Score : 0.8211

Classification Report:
              precision    recall  f1-score   support

 Not Churned       0.85      0.83      0.84      1035
     Churned       0.56      0.60      0.58       374

    accuracy                           0.77      1409
   macro avg       0.71      0.71      0.71      1409
weighted avg       0.77      0.77      0.77      1409



## Model 3: XGBoost — Production Model
Industry standard for tabular data.
Builds trees sequentially — each tree fixes previous tree's mistakes.

In [12]:
print("Training XGBoost...")

xgb_model = XGBClassifier(
    n_estimators=200,       # number of trees
    learning_rate=0.05,     # how much each tree corrects mistakes
    max_depth=5,            # how deep each tree can grow
    random_state=42,
    eval_metric='logloss',  # internal evaluation metric
    verbosity=0             # suppress training output
)
xgb_model.fit(X_train_bal, y_train_bal)

xgb_probs = xgb_model.predict_proba(X_test)[:, 1]
xgb_preds = xgb_model.predict(X_test)

xgb_auc = roc_auc_score(y_test, xgb_probs)

print(f"\n✓ XGBoost trained")
print(f"  ROC-AUC Score : {xgb_auc:.4f}")
print(f"\nClassification Report:")
print(classification_report(y_test, xgb_preds,
      target_names=['Not Churned', 'Churned']))

Training XGBoost...

✓ XGBoost trained
  ROC-AUC Score : 0.8291

Classification Report:
              precision    recall  f1-score   support

 Not Churned       0.87      0.82      0.84      1035
     Churned       0.56      0.65      0.60       374

    accuracy                           0.77      1409
   macro avg       0.72      0.73      0.72      1409
weighted avg       0.79      0.77      0.78      1409



In [13]:
print("=" * 45)
print("   MODEL COMPARISON SUMMARY")
print("=" * 45)
print(f"  {'Model':<25} {'ROC-AUC':>10}")
print(f"  {'-'*35}")
print(f"  {'Logistic Regression':<25} {lr_auc:>10.4f}")
print(f"  {'Random Forest':<25} {rf_auc:>10.4f}")
print(f"  {'XGBoost':<25} {xgb_auc:>10.4f}")
print("=" * 45)

best_auc = max(lr_auc, rf_auc, xgb_auc)
best_name = ['Logistic Regression','Random Forest','XGBoost'][
    [lr_auc, rf_auc, xgb_auc].index(best_auc)]
print(f"\n  Best model: {best_name} ({best_auc:.4f})")

   MODEL COMPARISON SUMMARY
  Model                        ROC-AUC
  -----------------------------------
  Logistic Regression           0.8205
  Random Forest                 0.8211
  XGBoost                       0.8291

  Best model: XGBoost (0.8291)


In [15]:
from sklearn.metrics import roc_curve, auc

fig, ax = plt.subplots(figsize=(8, 6))

# Define models, their probabilities and colors separately
models = [
    ('Logistic Regression', lr_probs,  '#378ADD'),
    ('Random Forest',       rf_probs,  '#1D9E75'),
    ('XGBoost',             xgb_probs, '#E24B4A'),
]

# Plot each model's ROC curve manually
for model_name, probs, color in models:
    fpr, tpr, _ = roc_curve(y_test, probs)
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, 
            color=color, 
            linewidth=2,
            label=f'{model_name} (AUC = {roc_auc:.3f})')

# Diagonal line = random guessing
ax.plot([0, 1], [0, 1], 
        'k--', linewidth=1,
        label='Random Guess (AUC = 0.500)')

ax.set_title('ROC Curves — Model Comparison',
              fontsize=14, fontweight='bold')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.legend(loc='lower right')
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.02])

plt.tight_layout()
plt.savefig('C:/Users/DELL/Documents/telecom-churn-analytics/reports/05_roc_curves.png',
            dpi=150, bbox_inches='tight')
plt.show()
plt.close()
print("✓ ROC curve saved to reports/05_roc_curves.png")

✓ ROC curve saved to reports/05_roc_curves.png


## SHAP: Why Did the Model Flag This Customer?

SHAP explains the model's predictions. For each feature it shows:
- Positive SHAP value = pushed prediction TOWARD churn
- Negative SHAP value = pushed prediction AWAY from churn

This is what makes the model usable by business teams.

In [16]:
print("Calculating SHAP values (this takes ~30 seconds)...")

# TreeExplainer works specifically with tree-based models
# like XGBoost and Random Forest
explainer = shap.TreeExplainer(xgb_model)

# Calculate SHAP values for the test set
# We use a sample of 500 rows to keep it fast
sample_idx = X_test.sample(500, random_state=42).index
X_test_sample = X_test.loc[sample_idx]
shap_values = explainer.shap_values(X_test_sample)

print("✓ SHAP values calculated")
print(f"  Shape of SHAP values: {shap_values.shape}")

Calculating SHAP values (this takes ~30 seconds)...
✓ SHAP values calculated
  Shape of SHAP values: (500, 25)


In [17]:
# Summary bar plot — shows which features matter most OVERALL
# Longer bar = feature has more influence on predictions
plt.figure(figsize=(10, 7))
shap.summary_plot(
    shap_values, 
    X_test_sample,
    plot_type='bar',
    show=False,
    max_display=15    # show top 15 features
)
plt.title('Top 15 Features by SHAP Importance', 
           fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('C:/Users/DELL/Documents/telecom-churn-analytics/reports/06_shap_importance.png',
            dpi=150, bbox_inches='tight')
plt.close()
print("✓ SHAP importance chart saved to reports/06_shap_importance.png")

✓ SHAP importance chart saved to reports/06_shap_importance.png


In [18]:
# Dot plot — shows DIRECTION of each feature's effect
# Red dots = high feature value pushing toward churn
# Blue dots = low feature value pushing away from churn
plt.figure(figsize=(10, 7))
shap.summary_plot(
    shap_values,
    X_test_sample,
    show=False,
    max_display=15
)
plt.title('SHAP Feature Impact — Direction and Magnitude',
           fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('C:/Users/DELL/Documents/telecom-churn-analytics/reports/07_shap_dot_plot.png',
            dpi=150, bbox_inches='tight')
plt.close()
print("✓ SHAP dot plot saved to reports/07_shap_dot_plot.png")

✓ SHAP dot plot saved to reports/07_shap_dot_plot.png


### How to Read the SHAP Charts

**Bar chart (06_shap_importance.png):**
The features with the longest bars have the most influence
on whether the model predicts churn or not.

**Dot chart (07_shap_dot_plot.png):**
- Each dot = one customer from the test set
- Position on x-axis = how much this feature pushed the prediction
- Color: Red = high feature value, Blue = low feature value

**Expected top features:**
- tenure (low tenure → more likely to churn)
- MonthlyCharges (high charges → more likely to churn)
- Contract_Two year (having two-year contract → less likely to churn)

This confirms our Day 1 EDA findings using a completely 
different analytical approach — which strengthens our conclusions.

## At-Risk Customer List
Export customers with churn probability > 0.5 who haven't 
churned yet. This is the actionable output for the business 
— the retention team can call these customers proactively.

In [19]:
# Get churn probabilities for ALL customers (not just test set)
# We use the full dataset to find all currently active at-risk customers
all_probs = xgb_model.predict_proba(X)[:, 1]

# Add probabilities back to original dataframe
df_risk = df.copy()
df_risk['churn_probability'] = all_probs
df_risk['risk_segment'] = pd.cut(
    df_risk['churn_probability'],
    bins=[0, 0.3, 0.5, 0.7, 1.0],
    labels=['Low Risk', 'Medium Risk', 'High Risk', 'Critical Risk']
)

# Filter: active customers (not yet churned) with probability > 0.5
at_risk = df_risk[
    (df_risk['Churn'] == 0) &           # not yet churned
    (df_risk['churn_probability'] > 0.5) # model says likely to churn
].copy()

# Sort by highest risk first
at_risk = at_risk.sort_values(
    'churn_probability', ascending=False
)

print(f"✓ At-risk customer analysis complete")
print(f"\n  Total active customers    : {(df_risk['Churn']==0).sum():,}")
print(f"  Flagged as at-risk (>50%) : {len(at_risk):,}")
print(f"  High Risk (50-70%)        : {(at_risk['risk_segment']=='High Risk').sum():,}")
print(f"  Critical Risk (70-100%)   : {(at_risk['risk_segment']=='Critical Risk').sum():,}")
print(f"\n  Monthly revenue at risk   : ${at_risk['MonthlyCharges'].sum():,.0f}")

✓ At-risk customer analysis complete

  Total active customers    : 5,174
  Flagged as at-risk (>50%) : 770
  High Risk (50-70%)        : 457
  Critical Risk (70-100%)   : 313

  Monthly revenue at risk   : $55,371


In [20]:
# Select key columns for the export
export_cols = [
    'customerID', 'tenure', 'Contract',
    'MonthlyCharges', 'TotalCharges',
    'churn_probability', 'risk_segment'
]

# Keep only columns that exist
export_cols = [c for c in export_cols if c in at_risk.columns]

at_risk_export = at_risk[export_cols].reset_index(drop=True)

# Save to reports folder for Power BI
at_risk_export.to_csv(
    'C:/Users/DELL/Documents/telecom-churn-analytics/reports/at_risk_customers.csv', 
    index=False
)

print(f"✓ At-risk list saved to reports/at_risk_customers.csv")
print(f"  Rows exported : {len(at_risk_export):,}")
print(f"\nTop 10 highest-risk active customers:")
print(at_risk_export.head(10).to_string(index=False))

✓ At-risk list saved to reports/at_risk_customers.csv
  Rows exported : 770

Top 10 highest-risk active customers:
customerID  tenure  MonthlyCharges  TotalCharges  churn_probability  risk_segment
5542-TBBWB       1           69.90         69.90           0.952073 Critical Risk
7439-DKZTW       1           80.55         80.55           0.946530 Critical Risk
7577-SWIFR       1           89.25         89.25           0.941469 Critical Risk
4060-LDNLU       7           96.20        639.70           0.937632 Critical Risk
7465-ZZRVX       1           70.35         70.35           0.935642 Critical Risk
1452-VOQCH       1           75.10         75.10           0.935205 Critical Risk
4912-PIGUY       1           84.60         84.60           0.931489 Critical Risk
8161-QYMTT       7           94.10        701.30           0.923680 Critical Risk
1640-PLFMP       1           70.25         70.25           0.922304 Critical Risk
2262-SLNVK       1           70.10         70.10           0.9215

In [24]:
# Save the trained XGBoost model as a .pkl file
# .pkl = pickle format = Python's way of saving any object to disk
model_path = 'C:/Users/DELL/Documents/telecom-churn-analytics/models/xgb_churn_model.pkl'
joblib.dump(xgb_model, model_path)
print(f"✓ Model saved to {model_path}")

# Also save the scaler (needed if we ever score new customers)
scaler_path = 'C:/Users/DELL/Documents/telecom-churn-analytics/models/scaler.pkl'
joblib.dump(scaler, scaler_path)
print(f"✓ Scaler saved to {scaler_path}")

# Final summary
print(f"\n{'='*50}")
print(f"      ML Pipeline Summary")
print(f"{'='*50}")
print(f"  Logistic Regression ROC-AUC : {lr_auc:.4f}")
print(f"  Random Forest ROC-AUC       : {rf_auc:.4f}")
print(f"  XGBoost ROC-AUC             : {xgb_auc:.4f}")
print(f"\n  At-risk customers found     : {len(at_risk_export):,}")
print(f"  Monthly revenue at risk     : ${at_risk['MonthlyCharges'].sum():,.0f}")
print(f"\n  Files saved:")
print(f"  ✓ models/xgb_churn_model.pkl")
print(f"  ✓ models/scaler.pkl")
print(f"  ✓ reports/05_roc_curves.png")
print(f"  ✓ reports/06_shap_importance.png")
print(f"  ✓ reports/07_shap_dot_plot.png")
print(f"  ✓ reports/at_risk_customers.csv")
print(f"{'='*50}")

✓ Model saved to C:/Users/DELL/Documents/telecom-churn-analytics/models/xgb_churn_model.pkl
✓ Scaler saved to C:/Users/DELL/Documents/telecom-churn-analytics/models/scaler.pkl

      ML Pipeline Summary
  Logistic Regression ROC-AUC : 0.8205
  Random Forest ROC-AUC       : 0.8211
  XGBoost ROC-AUC             : 0.8291

  At-risk customers found     : 770
  Monthly revenue at risk     : $55,371

  Files saved:
  ✓ models/xgb_churn_model.pkl
  ✓ models/scaler.pkl
  ✓ reports/05_roc_curves.png
  ✓ reports/06_shap_importance.png
  ✓ reports/07_shap_dot_plot.png
  ✓ reports/at_risk_customers.csv


In [25]:
import os
report_files = sorted(os.listdir('C:/Users/DELL/Documents/telecom-churn-analytics/reports'))
model_files = sorted(os.listdir('C:/Users/DELL/Documents/telecom-churn-analytics/models'))

print("Files in reports/:")
for f in report_files:
    size = os.path.getsize(f'C:/Users/DELL/Documents/telecom-churn-analytics/reports/{f}') / 1024
    print(f"  ✓ {f} ({size:.1f} KB)")

print(f"\nFiles in models/:")
for f in model_files:
    if not f.startswith('.'):
        size = os.path.getsize(f'C:/Users/DELL/Documents/telecom-churn-analytics/models/{f}') / 1024
        print(f"  ✓ {f} ({size:.1f} KB)")

Files in reports/:
  ✓ 01_churn_distribution.png (74.4 KB)
  ✓ 02_churn_by_contract.png (37.1 KB)
  ✓ 03_tenure_by_churn.png (52.3 KB)
  ✓ 04_charges_by_churn.png (32.3 KB)
  ✓ 05_roc_curves.png (102.7 KB)
  ✓ 06_shap_importance.png (104.5 KB)
  ✓ 07_shap_dot_plot.png (161.2 KB)
  ✓ at_risk_customers.csv (36.7 KB)

Files in models/:
  ✓ scaler.pkl (2.0 KB)
  ✓ xgb_churn_model.pkl (447.2 KB)
